# Intelligence Map RAG Platform - Colab One-Click Deployment

這個筆記本將協助您在 Google Colab 上快速佈署 Intelligence Map RAG 專案。

## 步驟：
1. **執行第一格**：安裝 Ollama 與專案依賴。
2. **執行第二格**：設定 ngrok 並啟動服務。

---

### 1. 環境準備與安裝

In [ ]:
# 1. 下載專案程式碼
import os
if not os.path.exists('MAP_RAG_MVP'):
    !git clone https://github.com/rgatsai/MAP_RAG_MVP.git
%cd MAP_RAG_MVP

# 2. 安裝 Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. 安裝 Python 依賴
!pip install -r requirements.txt pyngrok

# 4. 啟動 Ollama 服務 (背景執行)
import subprocess
import time

print("正在識別 Ollama 安裝路徑...")
ollama_path = "/usr/local/bin/ollama"
if not os.path.exists(ollama_path):
    try:
        ollama_path = subprocess.check_output(["which", "ollama"]).decode().strip()
    except:
        ollama_path = "ollama"

print(f"正在啟動 Ollama (路徑: {ollama_path})...")
try:
    subprocess.Popen([ollama_path, 'serve'], 
                     stdout=open('ollama.log', 'w'), 
                     stderr=open('ollama.err', 'w'))
    time.sleep(15) # 等待 15 秒啟動
    print("Ollama 背景服務啟動指令已送出。")
except Exception as e:
    print(f"❌ 啟動錯誤: {e}")

# 5. 下載模型
print("正在下載 Qwen 2.5 7B 模型 (這可能需要幾分鐘)... ")
!{ollama_path} pull qwen2.5:7b

### 2. 設定 ngrok 並啟動 API

> **注意**：您需要一個 [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) 的 Authtoken 才能公開存取 API。

In [ ]:
from pyngrok import ngrok
import getpass

# 輸入 ngrok token
print("請到 https://dashboard.ngrok.com/get-started/your-authtoken 複製您的 Token")
authtoken = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(authtoken)

# 建立穿透端口 8000
try:
    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 您的 API 已在以下網址公開: {public_url}")
    print("您可以開始向這個網址發送請求了！\n")
except Exception as e:
    print(f"❌ 穿透連線失敗: {e}")

# 啟動 FastAPI 服務
!uvicorn main:app --host 0.0.0.0 --port 8000